In [1]:
import numpy as np
import cvxpy as cp
import mosek
import random
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [3]:
def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def h_3(x,alfa):
    return(min(1,x/(1-alfa)))

def ranktoset (A):
    A = list(A)
    sets = [[A[0]]]
    for i in range(1,len(A)):
        new = A[0:i+1]
        sets.append(new)
    return(sets)


In [4]:
def robustcheck(a,R,r,p,m,r_f):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    #extra = 0
    #if np.min(x) < 0:
        #extra = np.min(x)
        #x = x - np.min(x)
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1, cp.sum(q_b)==1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons -cp.entr(q[i]) - q[i]*np.log(p[i])
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    print(prob.value - (1-np.sum(a))*r_f)#+extra)
    return(prob.value,q.value,q_b.value)

In [5]:
def dual (sets,p,R,r,m,r_f,a):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    v = cp.Variable((M,N))
    lbda = cp.Variable(M, nonneg = True)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N, nonneg = True)
    z2 = 0
    z4 = 0
    constraints = []
    for i in range(N):
        lbdasum = 0
        for j in range(M):
            if i in sets[j]:
                constraints.append(v[j][i] >= 0)
                lbdasum = lbdasum + lbda[j]
            else:
                constraints.append(v[j][i] >= 0)
        constraints.append((-R.dot(a))[i]-(1-sum(a))*r_f - lbdasum - beta <= 0)
        z4 = z4 + p[i]*t[i]
        constraints.append(-alpha + cp.sum(v[0:M:1,i]) + cp.kl_div(gamma, t[i]) + gamma - t[i] <= 0)
    for j in range(M):
        z1 = -cp.min(v[j,sets[j]])*(1-m)+lbda[j]
        z2 = z2 + cp.pos(z1)
    obj= cp.Minimize(alpha + beta + gamma * (r-1) + z4 + z2)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value,v.value,lbda.value,alpha.value,beta.value,gamma.value,t.value)

In [6]:
np.random.seed(10)

In [14]:
N=100
p = (np.zeros(N)+1)*1/N
I = 2
R = np.random.normal(0.05,0.2,size=(N,I))
print(R.transpose().dot(p))


[0.07115874 0.04091447]


In [18]:
a = np.zeros(I)+1/I
r = 0.043
m = 0.95
r_f = 0.001
robustcheck(a,R,r,p,m,r_f)

0.25511492454435286


(0.25511492454435286,
 array([0.00959174, 0.00960396, 0.00960235, 0.00956892, 0.00960421,
        0.00956426, 0.05000607, 0.00956584, 0.00957396, 0.00958994,
        0.00959087, 0.00959258, 0.00963408, 0.00959491, 0.00957838,
        0.00964224, 0.00958901, 0.00958281, 0.00959842, 0.00959339,
        0.00960199, 0.00956528, 0.0095839 , 0.00959953, 0.00961586,
        0.00957505, 0.00960288, 0.00961689, 0.00959957, 0.00956449,
        0.0095665 , 0.00959999, 0.00962102, 0.00960006, 0.00960012,
        0.00963694, 0.00957726, 0.00959565, 0.00964263, 0.00956805,
        0.00963576, 0.00960893, 0.00963946, 0.00959712, 0.00957083,
        0.00962795, 0.00960322, 0.00957184, 0.00956617, 0.00960089,
        0.00956962, 0.00959899, 0.00964425, 0.00960646, 0.00959639,
        0.00963792, 0.00960423, 0.00959416, 0.0096364 , 0.00959974,
        0.00956482, 0.0095647 , 0.00956985, 0.00963915, 0.00960332,
        0.00960284, 0.0096041 , 0.00956438, 0.00959787, 0.00964651,
        0.00958803, 0.0095

In [11]:
        
x=np.arange(1,N)
psets = list(powerset(list(range(N))))
for i in range(1,len(psets)):
    psets[i] = list(psets[i])
psets = psets[1:(len(psets))]

In [17]:
rank = np.argsort(R.dot(a))
sets = ranktoset(rank)
dual (sets,p,R,r,m,r_f,a)

(0.25511492457404733,
 array([[-2.78488536e-15,  2.40698819e-14,  2.79458059e-14, ...,
          1.78412545e-14, -6.60656656e-15,  1.07887091e-14],
        [-0.00000000e+00, -0.00000000e+00, -0.00000000e+00, ...,
         -0.00000000e+00, -0.00000000e+00, -0.00000000e+00],
        [-0.00000000e+00, -0.00000000e+00, -0.00000000e+00, ...,
         -0.00000000e+00, -0.00000000e+00, -0.00000000e+00],
        ...,
        [ 1.34777718e-14,  3.92779826e-14,  4.27292789e-14, ...,
          3.36622422e-14,  9.70735097e-15,  2.69916024e-14],
        [ 1.34209377e-14,  3.86327494e-14,  4.20841988e-14, ...,
          3.30101848e-14,  9.57272596e-15,  2.70460382e-14],
        [ 1.26793443e-14,  3.86136846e-14,  4.20768031e-14, ...,
          3.29550790e-14,  8.73517467e-15,  2.63924592e-14]]),
 array([0.05127066, 0.02359632, 0.015615  , 0.01117794, 0.00902176,
        0.00753945, 0.00657515, 0.00592146, 0.00549549, 0.00524349,
        0.005015  , 0.00487626, 0.00466564, 0.00446534, 0.00428479,
   

In [54]:
[probv,vv,lbdav,alphav,betav,gammav,tv]=dual (sets,p,R,r,m,r_f,a)
N = len(p)
M = len(sets)
cons1 =np.zeros(N)
cons2 = np.zeros(N)
z0 = 0
for j in range(M):
    z9 = -np.min(vv[j,sets[j]])*(1-m)+lbdav[j]
    z0 = z0 + max(z9,0)
for i in range(N):
    lbdsom = 0
    for j in range(M):
        if i in sets[j]:
            lbdsom = lbdsom + lbdav[j]
    cons1[i] = R.dot(a)[i] + betav + lbdsom
    cons2[i] = gammav * np.exp((-alphav+sum(vv[0:M:1,i]))/gammav)-tv[i]
print(cons1)
print(cons2)
print(-1+alphav+betav+gammav*r+sum(p*tv)+z0)

[ 5.70003067e-09 -5.38403810e-09  6.53539289e+00  1.70622106e+01]
[-8.10329546e-08 -2.98001260e-06 -3.94966149e-08 -2.06583066e-08]
[24.65265508]
